In [ ]:
import pathlib
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
from prepare import process_regressor, model_performance_metrics,draw_scatter
from data_prune_function import get_scores,get_data,prune_rd,return_model
random_state =352
test_size = 0.1

model = {}
for modelname in ['CB','XG','GB','RF']:
    model[modelname] = return_model(modelname,random_state=random_state)

target = 'NO(%)'
df,X,y,X_train_val,X_test,y_train_val,y_test = get_data(target = target,random_state=random_state, test_size=test_size,standardized=False)
mad = (y - y.mean()).abs().mean()
std = y.std()
print('')
print('-----------------')
print(f'{target} {y.shape[0]} {mad:.3f} {std:.3f} ')
print('-----------------')

In [ ]:
target = 'NO(%)'
a = 'CB'
b = 'XG'
c = 'GB'
d = 'RF'

In [ ]:
folder = f'{target}/{target}_{a}_guiding_pruning'
pathlib.Path(f"./{folder}").mkdir(parents=True, exist_ok=True)
file_out_a = f'{folder}/all_dat.pkl'

folder = f'{target}/{target}_{b}_guiding_pruning'
pathlib.Path(f"./{folder}").mkdir(parents=True, exist_ok=True)
file_out_b = f'{folder}/all_dat.pkl'

folder = f'{target}/{target}_{c}_guiding_pruning'
pathlib.Path(f"./{folder}").mkdir(parents=True, exist_ok=True)
file_out_c = f'{folder}/all_dat.pkl'

In [ ]:
train_pred, test_pred = process_regressor(model[a],X_train_val,X_test,y_train_val,y_test)
model_performance_metrics('Cat Train', y_train_val, train_pred)
model_performance_metrics('Cat Test', y_test, test_pred)
draw_scatter(y_train_val, train_pred, y_test, test_pred, "Catboost", actual_text='Actual NO removal (%)',pred_text='Predicted NO removal (%)')

In [ ]:
train_pred, test_pred = process_regressor(model[b],X_train_val,X_test,y_train_val,y_test)
model_performance_metrics('XGB Train', y_train_val, train_pred)
model_performance_metrics('XGB Test', y_test, test_pred)
draw_scatter(y_train_val, train_pred, y_test, test_pred, "XGBoost", actual_text='Actual NO removal (%)',pred_text='Predicted NO removal (%)',save=True)

In [ ]:
train_pred, test_pred = process_regressor(model[c],X_train_val,X_test,y_train_val,y_test)
model_performance_metrics('GBDT Train', y_train_val, train_pred)
model_performance_metrics('GBDT Test', y_test, test_pred)
draw_scatter(y_train_val, train_pred, y_test, test_pred, "GradientBoost", actual_text='Actual NO removal (%)',pred_text='Predicted NO removal (%)',save=False)

In [ ]:
train_pred, test_pred = process_regressor(model[c],X_train_val,X_test,y_train_val,y_test)
model_performance_metrics('RF Train', y_train_val, train_pred)
model_performance_metrics('RF Test', y_test, test_pred)
draw_scatter(y_train_val, train_pred, y_test, test_pred, "RF", actual_text='Actual NO removal (%)',pred_text='Predicted NO removal (%)',save=False)

In [ ]:
maes_a, rmse_a, r2_a = get_scores(model[a], X_train_val, y_train_val, X_test, y_test)

In [ ]:
maes_b, rmse_b, r2_b = get_scores(model[b], X_train_val, y_train_val, X_test, y_test)

In [ ]:
maes_c, rmse_c, r2_c = get_scores(model[c], X_train_val, y_train_val, X_test, y_test)

In [ ]:
size_old_val_a, ids_a, test_scores_a, val_scores_a,test_fit_a,remove_a = prune_rd(
        model[a], X_train_val, y_train_val, X_test, y_test,max_iter=80,
        min_drop= 2,model_test= model[d],
        threshold=maes_a / 2,train_size= 0.9, file_out=file_out_a)

In [ ]:
size_old_val_b, ids_b, test_scores_b, val_scores_b,test_fit_b,remove_b = prune_rd(
    model[b], X_train_val, y_train_val, X_test, y_test,max_iter=80,min_drop= 2,
    model_test= model[d],threshold=maes_b / 2,train_size= 0.9, file_out=file_out_b)

In [ ]:
size_old_val_c, ids_c, test_scores_c, val_scores_c,test_fit_c,remove_c = prune_rd(
        model[c], X_train_val, y_train_val, X_test, y_test,max_iter=80,
        min_drop= 2,model_test= model[d],threshold=maes_c / 2,train_size= 0.9, file_out=file_out_c)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from prepare import get_mse_rmse_mae_mape_smape_r2
import os
def draw_scatter_3(train_true, train_pred,val_true,val_pred,test_true, test_pred,title,
                  actual_text='actual', pred_text='prediction',save=False):


    plt.rcParams['font.family'] = 'Times New Roman'



    font_common = {
        'family': 'Times New Roman',
        'color': '#2F4F4F', 
        'weight': 'bold',
        'size': 18
    }


    sns.regplot(x=train_true, y=train_pred,
                color="dodgerblue", label= 'train',
                scatter_kws = {'alpha': 0.5, 's': 18})
    sns.regplot(x=test_true, y=test_pred, color= '#FF69B4', label= 'test',
                scatter_kws = {'alpha': 0.8, 's': 30})
    sns.regplot(x=val_true, y= val_pred,
                color="orange", label="val",
                scatter_kws = {'alpha': 0.5, 's': 10})

    plt.xlabel(actual_text, fontdict=font_common,weight='bold')
    plt.ylabel(pred_text, fontdict=font_common,weight='bold')
    plt.xlim(0, 100)
    plt.ylim(0, 100)


    legend_font = {'family': 'Times New Roman', 'weight': 'bold', 'size': 15}
    plt.legend(loc="upper right", markerscale=1.5, prop=legend_font,frameon=True, bbox_to_anchor=(0.3, 0.85))



    plt.tick_params(direction="in", width=2, labelsize=14)


    ax = plt.gca()
    
    ax.set_facecolor('white')

 
    for spine in ax.spines.values():
        spine.set_linewidth(2)
        spine.set_edgecolor('black')


    plt.grid(True,
             axis='both',
             linestyle='--',
             linewidth=0.5,
             color='gray',
             alpha=0.6)


    _, _, _, _, _, train_r2 = get_mse_rmse_mae_mape_smape_r2(train_true.values.ravel() / 100, train_pred / 100)
    _, _, _, _, _, val_r2 = get_mse_rmse_mae_mape_smape_r2(val_true.values.ravel() / 100, val_pred / 100)
    _, _, _, _, _, test_r2 = get_mse_rmse_mae_mape_smape_r2(test_true.values.ravel() / 100, test_pred / 100)

    plt.text(60, 26, f"Train R² = {(int(train_r2*1000)/1000):.3f}", **font_common)
    plt.text(60, 18, f"val   R² = {int(val_r2*1000)/1000:.3f}", **font_common)
    plt.text(60, 10, f"test  R² = {int(test_r2*1000)/1000:.3f}", **font_common)


    plt.text(50, 90, title, **font_common, horizontalalignment="center")

    if save:
        out_put = 'scatter_images'
        if not os.path.exists(out_put):
            os.makedirs(out_put)
        output_path = os.path.join(out_put, f'{title}.jpg')
        plt.savefig(output_path, format="jpg", dpi=600,
                    bbox_inches="tight",
                    pil_kwargs={'quality': 95})
        print(f"save fig：{output_path}")


    plt.show()

In [ ]:
from prepare import get_mse_rmse_mae_mape_smape_r2
import os

def draw_scatter_1(train_true, train_pred, val_true, val_pred, test_true, test_pred, title,
                  actual_text='actual', pred_text='prediction', save=False):

    

    

    plt.rcParams['font.family'] = 'Times New Roman'

    fig, ax = plt.subplots(figsize=(10, 8))
    

    font_common = {
        'family': 'Times New Roman',
        'color': '#2F4F4F',
        'weight': 'bold',
        'size': 18
    }
    
    legend_font = {'family': 'Times New Roman', 'weight': 'bold', 'size': 15}


    sns.regplot(x=train_true, y=train_pred,
                color="dodgerblue", label='train',
                scatter_kws={'alpha': 0.5, 's': 18},
                ax=ax)
    sns.regplot(x=test_true, y=test_pred, color='#FF69B4', label='test',
                scatter_kws={'alpha': 0.8, 's': 30},
                ax=ax)
    sns.regplot(x=val_true, y=val_pred,
                color="orange", label="val",
                scatter_kws={'alpha': 0.5, 's': 10},
                ax=ax)


    ax.set_xlabel(actual_text, fontdict=font_common, weight='bold')
    ax.set_ylabel(pred_text, fontdict=font_common, weight='bold')
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100)


    ax.legend(loc="upper right", markerscale=1.5, prop=legend_font, 
              frameon=True, bbox_to_anchor=(0.3, 0.85))


    ax.tick_params(axis='both', which='major', direction='out', 
                   width=2, labelsize=14,
                   bottom=True, top=False, left=True, right=False)  # 只显示下轴和左轴的主刻度
    

    ax.tick_params(axis='both', which='minor', direction='out',
                   bottom=True, top=False, left=True, right=False)
    
 
    ax.xaxis.set_major_locator(plt.MultipleLocator(20))  # x轴每20一个主刻度
    ax.yaxis.set_major_locator(plt.MultipleLocator(20))  # y轴每20一个主刻度
    

    ax.xaxis.set_minor_locator(plt.MultipleLocator(10))  # x轴每10一个次刻度
    ax.yaxis.set_minor_locator(plt.MultipleLocator(10))  # y轴每10一个次刻度


    ax.set_facecolor('white')
    

    for spine in ax.spines.values():
        spine.set_linewidth(2)
        spine.set_edgecolor('black')


    ax.grid(True, axis='both', linestyle='--', 
            linewidth=0.5, color='gray', alpha=0.6)


    _, _, _, _, _, train_r2 = get_mse_rmse_mae_mape_smape_r2(
        train_true.values.ravel() / 100, train_pred / 100)
    _, _, _, _, _, val_r2 = get_mse_rmse_mae_mape_smape_r2(
        val_true.values.ravel() / 100, val_pred / 100)
    _, _, _, _, _, test_r2 = get_mse_rmse_mae_mape_smape_r2(
        test_true.values.ravel() / 100, test_pred / 100)

    ax.text(60, 26, f"Train R² = {train_r2:.3f}", **font_common)
    ax.text(60, 18, f"val   R² = {val_r2:.3f}", **font_common)
    ax.text(60, 10, f"test  R² = {test_r2:.3f}", **font_common)


    ax.text(50, 90, title, **font_common, horizontalalignment="center")


    if save:
        out_put = 'scatter_images'
        if not os.path.exists(out_put):
            os.makedirs(out_put)
        output_path = os.path.join(out_put, f'{title}.jpg')
        plt.savefig(output_path, format="jpg", dpi=600,
                    bbox_inches="tight",
                    pil_kwargs={'quality': 95})
        print(f"save fig：{output_path}")

    plt.tight_layout()
    plt.show()

In [ ]:

a_test_90 = list(set(ids_a['train_val']) - set(remove_a[6]))
b_test_90 = list(set(ids_b['train_val']) - set(remove_b[8]))
c_test_90 = list(set(ids_c['train_val']) - set(remove_c[4]))



a_test_70 = list(set(ids_a['train_val']) - set(remove_a[22]))
b_test_70 = list(set(ids_b['train_val']) - set(remove_b[20]))
c_test_70 = list(set(ids_c['train_val']) - set(remove_c[20]))




a_test_50 = list(set(ids_a['train_val']) - set(remove_a[40]))
b_test_50 = list(set(ids_b['train_val']) - set(remove_b[46]))
c_test_50 = list(set(ids_c['train_val']) - set(remove_c[41]))


a_test_30 = list(set(ids_a['train_val']) - set(remove_a[62]))
b_test_30 = list(set(ids_b['train_val']) - set(remove_b[67]))
c_test_30 = list(set(ids_c['train_val']) - set(remove_c[63]))


In [ ]:
a_dict = {}
a_dict[6] = a_test_90
a_dict[22] = a_test_70
a_dict[40] = a_test_50
a_dict[62] = a_test_30

b_dict = {}
b_dict[8] = b_test_90
b_dict[20] = b_test_70
b_dict[46] = b_test_50
b_dict[67] = b_test_30

c_dict = {}
c_dict[4] = c_test_90
c_dict[20] = c_test_70
c_dict[41] = c_test_50
c_dict[63] = c_test_30

In [ ]:
df.loc[remove_a[0]].describe()

In [ ]:
df.loc[remove_a[6]].describe()

In [ ]:
df.loc[remove_a[20]].describe()

In [ ]:
df.loc[remove_a[40]].describe()

In [ ]:
df.loc[remove_a[62]].describe()

In [ ]:
df.loc[remove_b[8]].describe()

In [ ]:
df.loc[remove_b[24]].describe()

In [ ]:
df.loc[remove_b[46]].describe()

In [ ]:
df.loc[remove_b[67]].describe()

In [ ]:
df.loc[remove_c[4]].describe()

In [ ]:
df.loc[remove_c[20]].describe()

In [ ]:
df.loc[remove_c[41]].describe()

In [ ]:
df.loc[remove_c[63]].describe()

In [ ]:
a_count = 0
for i in a_dict.keys():
    df_a = df.loc[remove_a[i]]
    X_train_val = df_a.drop(columns=['NO(%)'])
    y_train_val = df_a['NO(%)']

    df_a_val = df.loc[a_dict[i]]
    X_val = df_a_val.drop(columns=['NO(%)'])
    y_val = df_a_val['NO(%)']

    train_pred, val_pred = process_regressor(model[a],X_train_val,X_val,y_train_val,y_val,cv=False)
    _,test_pred = process_regressor(model[a],X_train_val,X_test,y_train_val,y_test,cv=False)
    if a_count == 0:
        model_performance_metrics('Cat Train 90%', y_train_val, train_pred)
        model_performance_metrics('Cat val 10%', y_val, val_pred)
        model_performance_metrics('Cat val 10%', y_test, test_pred)
        draw_scatter_1(y_train_val, train_pred, y_val,val_pred,y_test, test_pred, "Catboost prune 90%", actual_text='Actual NO removal (%)',pred_text='Predicted NO removal (%)',save=True)

    if a_count == 1:
        model_performance_metrics('Cat Train 70%', y_train_val, train_pred)
        model_performance_metrics('Cat val 30%', y_val, val_pred)
        model_performance_metrics('Cat test', y_test, test_pred)
        draw_scatter_1(y_train_val, train_pred, y_val,val_pred,y_test, test_pred, "Catboost prune 70%", actual_text='Actual NO removal (%)',pred_text='Predicted NO removal (%)',save=True)
    if a_count == 2:
        model_performance_metrics('Cat Train 50%', y_train_val, train_pred)
        model_performance_metrics('Cat val 50%', y_val, val_pred)
        model_performance_metrics('Cat test', y_test, test_pred)
        draw_scatter_1(y_train_val, train_pred, y_val,val_pred,y_test, test_pred, "Catboost prune 50%", actual_text='Actual NO removal (%)',pred_text='Predicted NO removal (%)',save=True)
    if a_count == 3:
        model_performance_metrics('Cat Train 30%', y_train_val, train_pred)
        model_performance_metrics('Cat val 30%', y_val, val_pred)
        model_performance_metrics('Cat test', y_test, test_pred)
        draw_scatter_1(y_train_val, train_pred,y_val,val_pred, y_test, test_pred, "Catboost prune 30%", actual_text='Actual NO removal (%)',pred_text='Predicted NO removal (%)',save=True)
    a_count = a_count+1
    print('-------------------------------------------------------------------------------')


In [ ]:
b_count = 0

for i in b_dict.keys():
    df_b = df.loc[remove_b[i]]
    X_train_val = df_b.drop(columns=['NO(%)'])
    y_train_val = df_b['NO(%)']

    df_b_val = df.loc[b_dict[i]]
    X_val = df_b_val.drop(columns=['NO(%)'])
    y_val = df_b_val['NO(%)']
    train_pred, val_pred = process_regressor(model[b],X_train_val,X_val,y_train_val,y_val)
    _,test_pred = process_regressor(model[b],X_train_val,X_test,y_train_val,y_test)
    if b_count == 0:
        model_performance_metrics('XGB Train 90%', y_train_val, train_pred)
        model_performance_metrics('XGB val 10%', y_val, val_pred)
        model_performance_metrics('XGB val 10%', y_test, test_pred)
        draw_scatter_1(y_train_val, train_pred, y_val,val_pred,y_test, test_pred, "XGBboost prune 90%", actual_text='Actual NO removal (%)',pred_text='Predicted NO removal (%)',save=True)

    if b_count == 1:
        model_performance_metrics('XGB Train 70%', y_train_val, train_pred)
        model_performance_metrics('XGB val 30%', y_val, val_pred)
        model_performance_metrics('XGB test', y_test, test_pred)
        draw_scatter_1(y_train_val, train_pred, y_val,val_pred,y_test, test_pred, "XGBboost prune 70%", actual_text='Actual NO removal (%)',pred_text='Predicted NO removal (%)',save=True)
    if b_count == 2:
        model_performance_metrics('XGB Train 50%', y_train_val, train_pred)
        model_performance_metrics('XGB val 50%', y_val, val_pred)
        model_performance_metrics('XGB test', y_test, test_pred)
        draw_scatter_1(y_train_val, train_pred, y_val,val_pred,y_test, test_pred, "XGBboost prune 50%", actual_text='Actual NO removal (%)',pred_text='Predicted NO removal (%)',save=True)
    if b_count == 3:
        model_performance_metrics('XGB Train 30%', y_train_val, train_pred)
        model_performance_metrics('XGB val 30%', y_val, val_pred)
        model_performance_metrics('XGB test', y_test, test_pred)
        draw_scatter_1(y_train_val, train_pred,y_val,val_pred, y_test, test_pred, "XGBboost prune 30%", actual_text='Actual NO removal (%)',pred_text='Predicted NO removal (%)',save=True)
    b_count = b_count+1
    print('-------------------------------------------------------------------------------')


In [ ]:
c_count = 0
from sklearn import metrics
for i in c_dict.keys():

    df_c = df.loc[remove_c[i]]
    X_train_val = df_c.drop(columns=['NO(%)'])
    y_train_val = df_c['NO(%)']

    df_c_val = df.loc[c_dict[i]]
    X_val = df_c_val.drop(columns=['NO(%)'])
    y_val = df_c_val['NO(%)']

    train_pred, val_pred = process_regressor(model[c],X_train_val,X_val,y_train_val,y_val,cv=True)
    _,test_pred = process_regressor(model[c],X_train_val,X_test,y_train_val,y_test,cv=True)
    if c_count == 0:
        model_performance_metrics('GradientBoost Train 90%', y_train_val, train_pred)
        model_performance_metrics('GradientBoost val 10%', y_val, val_pred)
        model_performance_metrics('GradientBoost val 10%', y_test, test_pred)
        print(metrics.r2_score(y_test, test_pred))
        draw_scatter_1(y_train_val, train_pred, y_val,val_pred,y_test, test_pred, "GradientBoost prune 90%", actual_text='Actual NO removal (%)',pred_text='Predicted NO removal (%)',save=True)

    if c_count == 1:
        model_performance_metrics('GradientBoost Train 70%', y_train_val, train_pred)
        model_performance_metrics('GradientBoost val 30%', y_val, val_pred)
        model_performance_metrics('GradientBoost test', y_test, test_pred)
        draw_scatter_1(y_train_val, train_pred, y_val,val_pred,y_test, test_pred, "GradientBoost prune 70%", actual_text='Actual NO removal (%)',pred_text='Predicted NO removal (%)',save=True)
    if c_count == 2:
        model_performance_metrics('GradientBoost Train 50%', y_train_val, train_pred)
        model_performance_metrics('GradientBoost val 50%', y_val, val_pred)
        model_performance_metrics('GradientBoost test', y_test, test_pred)
        draw_scatter_1(y_train_val, train_pred, y_val,val_pred,y_test, test_pred, "GradientBoost prune 50%", actual_text='Actual NO removal (%)',pred_text='Predicted NO removal (%)',save=True)
    if c_count == 3:
        model_performance_metrics('GradientBoost Train 30%', y_train_val, train_pred)
        model_performance_metrics('GradientBoost val 30%', y_val, val_pred)
        model_performance_metrics('GradientBoost test', y_test, test_pred)
        draw_scatter_1(y_train_val, train_pred,y_val,val_pred, y_test, test_pred, "GradientBoost prune 30%", actual_text='Actual NO removal (%)',pred_text='Predicted NO removal (%)',save=True)
    c_count = c_count+1
    print('-------------------------------------------------------------------------------')


In [ ]:
full = {}
full['id'] = df.index.tolist()

In [ ]:
a_count = []
for i in range(0,80):
    aaaaaaa = {}
    aaaaaaa['id']= list(set(full['id']) - set(remove_a[i]))
    df_a = df.loc[remove_a[i]]
    X_train_val = df_a.drop(columns=['NO(%)'])
    y_train_val = df_a['NO(%)']

    df_a_val = df.loc[aaaaaaa['id']]
    X_val = df_a_val.drop(columns=['NO(%)'])
    y_val = df_a_val['NO(%)']
    train_pred, val_pred = process_regressor(model[a],X_train_val,X_val,y_train_val,y_val,cv=False)
    model_performance_metrics('Cat Train 90%', y_train_val, train_pred)
    abc = model_performance_metrics('Cat val 10%', y_val, val_pred)
    print('______-----------------------------------------')
    a_count.append(abc)

a_count = pd.DataFrame(a_count, columns=['ID'])
df_111 = a_count.sort_values(by = 'ID')
print(df_111)

In [ ]:
aaaaaaa = {}
aaaaaaa['id']= list(set(full['id']) - set(remove_a[20]))

In [ ]:
aaaaaaa = {}
aaaaaaa['id']= list(set(full['id']) - set(remove_a[22]))
df_a = df.loc[remove_a[22]]
df_a_val = df.loc[aaaaaaa['id']]

df_a.to_csv('prune_train.csv', index=False)
df_a_val.to_csv('prune_test.csv', index=False)

In [ ]:
a_count = []
for i in range(0,80):
    aaaaaaa = {}
    aaaaaaa['id']= list(set(full['id']) - set(remove_b[i]))
    df_a = df.loc[remove_b[i]]
    X_train_val = df_a.drop(columns=['NO(%)'])
    y_train_val = df_a['NO(%)']

    df_a_val = df.loc[aaaaaaa['id']]
    X_val = df_a_val.drop(columns=['NO(%)'])
    y_val = df_a_val['NO(%)']
    train_pred, val_pred = process_regressor(model[b],X_train_val,X_val,y_train_val,y_val,cv=False)
    model_performance_metrics('Cat Train 90%', y_train_val, train_pred)
    abc = model_performance_metrics('Cat val 10%', y_val, val_pred)
    print('______-----------------------------------------')
    a_count.append(abc)

a_count = pd.DataFrame(a_count, columns=['ID'])
df_111 = a_count.sort_values(by = 'ID')
print(df_111)

In [ ]:
a_count = []
for i in range(0,80):
    aaaaaaa = {}
    aaaaaaa['id']= list(set(full['id']) - set(remove_c[i]))
    df_a = df.loc[remove_c[i]]
    X_train_val = df_a.drop(columns=['NO(%)'])
    y_train_val = df_a['NO(%)']

    df_a_val = df.loc[aaaaaaa['id']]
    X_val = df_a_val.drop(columns=['NO(%)'])
    y_val = df_a_val['NO(%)']
    train_pred, val_pred = process_regressor(model[c],X_train_val,X_val,y_train_val,y_val,cv=False)
    model_performance_metrics('Cat Train 90%', y_train_val, train_pred)
    abc = model_performance_metrics('Cat val 10%', y_val, val_pred)
    print('______-----------------------------------------')
    a_count.append(abc)

a_count = pd.DataFrame(a_count, columns=['ID'])
df_111 = a_count.sort_values(by = 'ID')
print(df_111)

In [ ]:
X_train_val

In [ ]:
import pandas as pd
a_list = []
b_list = []
c_list = []
for i in range(0,80):
    a_list.append(len(remove_a[i]))
    b_list.append(len(remove_b[i]))
    c_list.append(len(remove_c[i]))
df_a = pd.DataFrame(a_list)
df_b = pd.DataFrame(b_list)
df_c = pd.DataFrame(c_list)

In [ ]:
test_scores_c = pd.DataFrame(test_scores_c)
val_scores_c = pd.DataFrame(val_scores_c)
test_fit_c = pd.DataFrame(test_fit_c)

test_scores_a = pd.DataFrame(test_scores_a)
val_scores_a = pd.DataFrame(val_scores_a)
test_fit= pd.DataFrame(test_fit_a)

test_scores_b = pd.DataFrame(test_scores_b)
val_scores_b = pd.DataFrame(val_scores_b)
test_fit_b = pd.DataFrame(test_fit_b)


In [ ]:

test_scores_a.to_csv('test_scores_a.csv')
val_scores_a.to_csv('val_scores_a.csv')
df_c.to_csv('df_c.csv')
df_a.to_csv('df_a.csv')
df_b.to_csv('df_b.csv')
test_scores_b.to_csv('test_scores_b.csv')
val_scores_b.to_csv('val_scores_b.csv')
test_fit_b.to_csv('test_fit_b.csv')
test_scores_c.to_csv('test_scores_c.csv')
val_scores_c.to_csv('val_scores_c.csv')
test_fit_c.to_csv('test_fit_c.csv')
test_fit.to_csv('test_fit.csv')

In [ ]:
df,X,y,X_train_val,X_test,y_train_val,y_test = get_data(target = target,random_state=random_state, test_size=test_size,standardized=False)

In [ ]:

df.loc[remove_a[20]].to_csv('70%.csv')
df.loc[a].to_csv('100%.csv')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
def draw_violin_1(data, config, which,model = '',bw_method='scott',count = '100%', save=False):


    plt.rcParams['font.family'] = 'Times New Roman'
    cfg = config[which]
    bw_method = cfg.get('bw_method', bw_method)


    data = data[which].values.ravel()
    scale_data = data * cfg['scale']



    min_val, max_val = scale_data.min(), scale_data.max()
    step = cfg['step'] * cfg['scale']

    scale_begin = max(0, int(min_val - step)) if cfg['begin'] < 0 else cfg['begin']
    scale_end = int(max_val + step + 1) if cfg['end'] < 0 else cfg['end']

    bins = range(scale_begin, scale_end, int(step))
    segments = pd.cut(scale_data, bins)

    mid_mapping = {
        interval: interval.mid / cfg['scale']
        for interval in segments.categories
    }


    mid_points = segments.rename_categories(mid_mapping).astype(float)


    plt.figure(figsize=cfg.get('figsize', (8, 6)))


    with sns.axes_style("whitegrid"):
        ax = sns.violinplot(
            x=mid_points,
            inner='box',
            bw_method=bw_method,
            color=cfg['color'],
            linewidth=cfg.get('linewidth', 1),
            saturation=cfg.get('saturation', 0.75),
            alpha = cfg.get('alpha',0.7)
        )


    plt.text(x = cfg['text_x'],y =cfg['text_y'],s = model +' '+count+' '+ cfg['text'] ,family='Times New Roman', weight='bold',
             size=(cfg['xtick_size']),
             horizontalalignment='center',
             transform=plt.gcf().transFigure,
             verticalalignment='top',)

    plt.xticks(fontproperties='Times New Roman', size=cfg['xtick_size'], weight='bold')


    plt.yticks(fontproperties='Times New Roman', size=20, weight='bold')


    plt.tick_params(direction="in", width=2)


    plt.ylim(cfg['ylim_left'], cfg['ylim_right'])


    ax = plt.gca()


    ax.set_xlabel(
    cfg['xlabel'],
    fontproperties='Times New Roman',  # 字体
    size=20,                           # 字体大小
    weight='bold',                     # 字体加粗
    labelpad=2                        # 标题与 x 轴的距离
                )

    ax.set_xticklabels(
    ax.get_xticklabels(),
    fontproperties='Times New Roman', 
    size=20,                           
    weight='bold',                  
    rotation=0,                     
    ha='right',                     
    va='top',                          
    )
    ax.xaxis.set_tick_params(pad=15)   

    ax.set_yticks([cfg['ylim_left'],cfg['ylim_left']/2,0,cfg['ylim_right']/2,cfg['ylim_right']])  # 主刻度位置
    ax.set_yticklabels([cfg['ylim_left'],cfg['ylim_left']/2,0,cfg['ylim_right']/2,cfg['ylim_right']])  # 主刻度标签

    for spine in ax.spines.values():
        spine.set_linewidth(cfg['bound_width'])
        spine.set_color('black')

    plt.grid(True,
             axis='both',
             linestyle='--',
             linewidth=0.5,
             color='gray',
             alpha=0.6)

    plt.subplots_adjust(top=0.9, bottom=0.1, left=0.1, right=0.9)



    if save:
        output_dir = 'violin_images'
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)
        output_path = os.path.join(output_dir, f'{cfg['image_name']+ model + count}.jpg')
        plt.savefig(output_path, format="jpg", dpi=600,
                    bbox_inches="tight",
                    pil_kwargs={'quality': 95})
        print(f"save fig：{output_path}")

    plt.show()

